<a href="https://colab.research.google.com/github/wesoho/Bringing-Old-Photos-Back-to-Life/blob/master/HyperFlow_H3_Flash_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 简单界面 + HyperFlow H3 Flash（免费 T4）· Colab 一键版

一个网页（提示词框 + 按钮）驱动 **HyperFlow H3 Flash** 出视频（**自带 32kHz 立体声音轨**），出片后页面内直接播放。附 REST API。**没有 ComfyUI 节点图**。

- 模型：HyperFlow H3 Flash = MiniMax-H3 的社区压缩版（Video Rebirth，33B→13.7B），**4/6/8 步无 CFG 采样**，GGUF Q4_K_M 仅 9.1GB；文本编码器与 VAE 沿用 MiniMax-H3 原版
- 后端：**三阶段批处理**（encode→sample→decode 各自独立进程），适配免费 T4 的 12.7G 主机内存；采样用官方 `HyperFlowLoader` + `HyperFlowSigmas`
- 出片：默认 512×384、2.3 秒（56 帧）、6 步，**一段约 25-45 分钟**（T4 未实测，为估算值；首次另需下载约 30G 权重）
- 接口：`POST /api/jobs`（头 `x-api-key`）→ 轮询 → 下载 / 页面内播放

流程：Cell 1（装引擎）→ Cell 2（下权重）→ Cell 3（起服务）→ Cell 4（拉起隧道域名，打印公网地址）→ 浏览器打开该地址，输入提示词点「开始生成」。**全笔记本没有任何自动演示/示例提交**：Cell 5 只是服务自检+防闲置保活，不提交任何生成任务，可跑可不跑；唯一出片路径就是网页里你自己点按钮。

调参（Cell 3 顶部改）：`GEAR_W/GEAR_H`（32 的倍数，T4 上别超过 768 宽，易 OOM）、`CLIP_SECONDS`（2.0→56 帧；5.2→124 帧，时间约 ×2.2；上限 15 秒）、`STEPS`（只能 4/6/8）。提示词里写上声音描述（雨声、人声、音乐）效果更好——模型会同步生成音轨。

注意：
- 免费会话 4-6 小时会断，**断了重跑 Cell 3、Cell 4 拿新地址**；浏览器用 **http://** 打开
- 权重为 MiniMax-H3 衍生（MiniMax H3 Community License，不含欧盟/英国/韩国/美国地区）；公开发布生成内容需注明「由 AI 生成」
- 我本地无 GPU 无法实测，跑起来报错把红字发我逐条修


In [ ]:
# @title 1. 安装 ComfyUI + GGUF/HyperFlow 节点 + 自建节点（3-5 分钟）
import os, sys, re, time, subprocess, socket, shutil, hashlib, threading, secrets
from pathlib import Path

# ---- 常量（所有 cell 共用）----
COMFY_DIR = Path("/content/ComfyUI")
MODEL_STORE = Path("/content/h3-store");   MODEL_STORE.mkdir(parents=True, exist_ok=True)
COND_STORE = Path("/content/h3-cond");      COND_STORE.mkdir(parents=True, exist_ok=True)
LATENT_STORE = Path("/content/h3-latent");  LATENT_STORE.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("/content/outputs");      OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMFY_LOG = Path("/content/comfyui.log")
COMFY_API = "http://127.0.0.1:8188"   # ComfyUI 后端地址（勿与其他变量撞名）

COMFY_COMMIT = "9a9fdb10ed144ce760d9682cb247526ea23cc525"      # 已验证：含 nodes_minimax_h3 / disable_dynamic / sample_sigmas 传链
GGUF_COMMIT = "6ea2651e7df66d7585f6ffee804b20e92fb38b8a"        # 已验证：含 mmap_released / pin_weight_to_device（disk 权重流式）
HYPERFLOW_COMMIT = "356649c32dca9a1dd46608d530ccb5e025fb338d"   # ComfyUI-HyperFlow-Flash v1.1.1

def _sh(cmd, cwd=None):
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)

def _pin(repo, dest, commit):
    if not dest.exists():
        _sh(["git", "clone", "--filter=blob:none", repo, str(dest)])
    _sh(["git", "-C", str(dest), "checkout", commit])

# ---- ComfyUI + GGUF + HyperFlow 节点（pin 到实测 commit）----
_pin("https://github.com/Comfy-Org/ComfyUI.git", COMFY_DIR, COMFY_COMMIT)
CUSTOM = COMFY_DIR / "custom_nodes"
_pin("https://github.com/city96/ComfyUI-GGUF.git", CUSTOM / "ComfyUI-GGUF", GGUF_COMMIT)
_pin("https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git",
     CUSTOM / "ComfyUI-HyperFlow-Flash", HYPERFLOW_COMMIT)

# （无需架构白名单补丁：ComfyUI-HyperFlow-Flash 自带无架构键 GGUF 读取，HyperFlowLoader 直接加载）

# ---- 依赖（含 API 服务所需）----
_sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_DIR / "requirements.txt")])
_sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(CUSTOM / "ComfyUI-GGUF" / "requirements.txt")])
_sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(CUSTOM / "ComfyUI-HyperFlow-Flash" / "requirements.txt")])
_sh([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "pydantic", "python-multipart", "requests"])

# ---- 写自建节点（encode/sample/decode 传 conditioning/latent）----
CTRL = CUSTOM / "H3BenchmarkControl"; CTRL.mkdir(parents=True, exist_ok=True)
RMSNORM_FP32 = True
(CTRL / "__init__.py").write_text(r'''
import json
import logging
import re
import sys
import time
from pathlib import Path

import torch

H3_TORCH_NESTED = "__h3_torch_nested__"
H3_COMFY_NESTED = "__h3_comfy_nested__"


def _is_comfy_nested(obj):
    """comfy.nested_tensor.NestedTensor, duck-typed.

    MiniMaxH3ImageToVideo returns its LATENT as this class rather than a torch
    nested tensor, so isinstance(obj, torch.Tensor) does not catch it.
    """
    return (not isinstance(obj, torch.Tensor)
            and bool(getattr(obj, "is_nested", False))
            and hasattr(obj, "unbind"))


def _restore(obj):
    if isinstance(obj, dict):
        keys = set(obj.keys())
        if keys == {H3_TORCH_NESTED}:
            return torch.nested.nested_tensor(list(obj[H3_TORCH_NESTED]))
        if keys == {H3_COMFY_NESTED}:
            from comfy.nested_tensor import NestedTensor
            return NestedTensor(list(obj[H3_COMFY_NESTED]))
        return {key: _restore(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_restore(value) for value in obj)
    if isinstance(obj, list):
        return [_restore(value) for value in obj]
    return obj


def _summarize(obj):
    if _is_comfy_nested(obj):
        return {"kind": "comfy_nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {"kind": "nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
        return {"kind": "tensor", "shape": list(obj.shape), "dtype": str(obj.dtype),
                "device": str(obj.device),
                "finite": bool(torch.isfinite(obj).all().item())
                if obj.is_floating_point() else None}
    if isinstance(obj, dict):
        return {key: _summarize(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_summarize(value) for value in obj]
    return {"kind": type(obj).__name__, "repr": repr(obj)[:200]}


def _detach_to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {H3_TORCH_NESTED: [p.detach().cpu() for p in obj.unbind()]}
        return obj.detach().cpu()
    if _is_comfy_nested(obj):
        return {H3_COMFY_NESTED: [p.detach().cpu() for p in obj.unbind()]}
    if isinstance(obj, dict):
        return {key: _detach_to_cpu(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_detach_to_cpu(value) for value in obj)
    if isinstance(obj, list):
        return [_detach_to_cpu(value) for value in obj]
    return obj


class H3SaveConditioning:
    """Write the encoder's output so later runs never load the encoder.

    Present here only for the regeneration path in Stage 2: the sampling graph
    itself must not contain a CLIPLoader.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "conditioning": ("CONDITIONING",), "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ()
    FUNCTION = "save"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save(self, conditioning, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        payload = {"schema_version": 1,
                   "conditioning": _detach_to_cpu(conditioning),
                   "samples": _detach_to_cpu(samples)}
        destination = directory / f"{key}.pt"
        staged = destination.with_suffix(".pt.tmp")
        torch.save(payload, staged)
        staged.replace(destination)
        (directory / f"{key}.summary.json").write_text(json.dumps(
            {"key": key, "bytes": destination.stat().st_size,
             "conditioning": _summarize(payload["conditioning"]),
             "samples": _summarize(payload["samples"])}, indent=2), encoding="utf-8")
        logging.info("H3_SAVE_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": destination.stat().st_size}, separators=(",", ":")))
        return {}


class H3LoadConditioning:
    """Return a precomputed CONDITIONING/LATENT pair without a CLIPLoader.

    This is what keeps the 14.61 GiB text encoder out of the sampling graph.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"}),
        }}
    RETURN_TYPES = ("CONDITIONING", "LATENT")
    RETURN_NAMES = ("conditioning", "samples")
    FUNCTION = "load"
    CATEGORY = "H3 benchmark/control"

    def load(self, store_dir, key):
        source = Path(store_dir) / f"{key}.pt"
        if not source.is_file():
            raise RuntimeError(f"Precomputed conditioning is missing: {source}")
        payload = torch.load(source, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError(f"Unsupported schema: {payload.get('schema_version')}")
        logging.info("H3_LOAD_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": source.stat().st_size}, separators=(",", ":")))
        return (_restore(payload["conditioning"]), _restore(payload["samples"]))


class H3LatentProbe:
    """Record the sampled latent without decoding it.

    Decoding needs the VAEs, and this stage is about the DiT alone.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "filename_prefix": ("STRING", {"default": "h3_probe/probe"}),
        }}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save_summary"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save_summary(self, samples, filename_prefix):
        import folder_paths
        summary = _summarize(samples)
        packed = samples.get("samples")
        parts = packed.unbind() if (_is_comfy_nested(packed)
                                    or getattr(packed, "is_nested", False)) else [packed]
        finite = [bool(torch.isfinite(p).all().item()) for p in parts
                  if isinstance(p, torch.Tensor)]
        # Write the report BEFORE raising: on a non-finite run the trace is the
        # whole point of the run, and an exception here must not discard it.
        output_path = Path(folder_paths.get_output_directory()) / f"{filename_prefix}.latent_summary.json"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {"summary": summary,
                   "shapes": [list(p.shape) for p in parts if isinstance(p, torch.Tensor)],
                   "finite": finite,
                   "nan_trace": _trace_report(),
                   "perf": _perf_report()}
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        logging.info("H3_LATENT_PROBE %s", json.dumps(payload["shapes"], separators=(",", ":")))
        if payload["nan_trace"] is not None:
            logging.info("H3_NAN_TRACE %s", json.dumps(
                payload["nan_trace"], separators=(",", ":")))
        if payload["perf"] is not None:
            logging.info("H3_PERF_REPORT %s", json.dumps(
                payload["perf"], separators=(",", ":")))
        if not finite or not all(finite):
            raise RuntimeError(f"Sampled latent contains non-finite values: {summary}")
        return (samples,)


# --- locating the first non-finite value -------------------------------------
# fp16 sampling produces a non-finite latent and upcasting RMSNorm did not fix it,
# so the overflow is somewhere else. Rather than patch by guesswork, hook every
# submodule of the DiT and record the FIRST one whose output goes non-finite while
# its input was still finite. That module is where the value is created; everything
# after it is downstream noise, so recording stops there.
#
# Module-level hooks are enough to answer the question that matters. The fused rope
# path is a function call inside Attention, not a module, so "Attention output bad,
# Attention input fine" localises it to the fused kernel without hooking the kernel.
#
# Cost: one GPU sync per module until the first bad one. At one step that is fine;
# it is not something to leave on for a real run.
H3_NAN_TRACE = {"armed": False, "done": False, "order": 0, "trail": [],
                "trail_len": 16, "culprit": None, "hooks": 0, "error": None,
                "skipped": 0, "errors": []}


def _trace_first_tensor(obj, depth=0):
    if isinstance(obj, torch.Tensor):
        return obj if obj.is_floating_point() else None
    if depth > 2:
        return None
    if isinstance(obj, (list, tuple)):
        for item in obj:
            found = _trace_first_tensor(item, depth + 1)
            if found is not None:
                return found
        return None
    if isinstance(obj, dict):
        return _trace_first_tensor(list(obj.values()), depth + 1)
    inner = getattr(obj, "tensors", None)           # comfy.nested_tensor.NestedTensor
    if isinstance(inner, (list, tuple)):
        return _trace_first_tensor(inner, depth + 1)
    return None


def _trace_stats(tensor):
    """min and max, one sync, no full-size temporary.

    The first version built an isfinite mask and an abs() copy, which is three
    extra tensors the size of the activation. On a 44232x5376 fp16 activation that
    is about a gigabyte, and it pushed a 14.9 GiB card into OOM mid-trace. aminmax
    is a reduction: it allocates two scalars.

    min/max propagate NaN, and inf survives them, so two numbers answer all three
    questions. The cost is that absmax is the max over ALL entries rather than over
    the finite ones -- which only matters on the tensor that already went bad.
    """
    low, high = torch.aminmax(tensor.detach())
    low, high = torch.stack([low.to(torch.float32),
                             high.to(torch.float32)]).tolist()
    has_nan = low != low or high != high          # NaN is the only value unequal to itself
    infinity = float("inf")
    has_inf = not has_nan and (low == -infinity or high == infinity)
    finite = not has_nan and not has_inf
    return {"finite": finite, "has_nan": has_nan, "has_inf": has_inf,
            "absmax": round(max(abs(low), abs(high)), 4) if finite else None,
            "min": round(low, 4) if finite else None,
            "max": round(high, 4) if finite else None}


def _trace_hook(name):
    def hook(module, args, output):
        state = H3_NAN_TRACE
        if state["done"]:
            return
        try:
            out = _trace_first_tensor(output)
            if out is None or out.numel() == 0:
                return
            state["order"] += 1
            stats = _trace_stats(out)
            record = {"order": state["order"], "module": name or "<root>",
                      "cls": type(module).__name__, "dtype": str(out.dtype),
                      "out": stats}
            state["trail"].append(record)
            if len(state["trail"]) > state["trail_len"]:
                state["trail"].pop(0)
            if not stats["finite"]:
                inp = _trace_first_tensor(args)
                record["in"] = (_trace_stats(inp) if inp is not None and inp.numel()
                                else None)
                state["culprit"] = record
                state["done"] = True    # everything downstream is just contaminated
        except Exception as exc:
            # Skip this module and keep going. An earlier version stopped the whole
            # trace on the first error, so one OOM inside the instrumentation threw
            # away the answer the run existed to produce.
            state["errors"] = state.get("errors", [])
            if len(state["errors"]) < 8:
                state["errors"].append(f"{name}: {type(exc).__name__}: "
                                       f"{str(exc).splitlines()[0][:160]}")
            state["skipped"] = state.get("skipped", 0) + 1
    return hook


def _trace_report():
    state = H3_NAN_TRACE
    if not state["armed"]:
        return None
    return {"armed": True, "hooks": state["hooks"], "modules_seen": state["order"],
            "culprit": state["culprit"], "trail": state["trail"],
            "error": state["error"], "skipped": state.get("skipped", 0),
            "errors": state.get("errors", []),
            "note": ("culprit is the first module whose output went non-finite; "
                     "in.finite=true there means the value is created inside it")}


class H3SaveLatent:
    """Persist the sampled AV latent so decoding can happen in another process.

    The DiT is 10.77 GiB and the video VAE is 4.85 GiB. Decoding in the same
    ComfyUI that just sampled would put both through a 12.7 GiB host, which is
    the sum the three-stage split exists to avoid. So the latent goes to disk and
    a fresh process picks it up.

    This runs BEFORE H3LatentProbe, so a non-finite latent is still written --
    a failed run is exactly when you want the artefact to look at.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": "/content/h3_sample_probe/latents"}),
            "key": ("STRING", {"default": "latent"}),
        }}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save(self, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        path = directory / f"{key}.pt"
        torch.save({"schema_version": 1, "key": key,
                    "samples": _detach_to_cpu(samples)}, path)
        logging.info("H3_SAVE_LATENT %s", json.dumps(
            {"key": key, "path": str(path), "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class H3LoadLatent:
    """Read back what H3SaveLatent wrote, in a process with no DiT in it."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": "/content/h3_sample_probe/latents"}),
            "key": ("STRING", {"default": "latent"}),
        }}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "load"
    CATEGORY = "H3 benchmark/control"

    def load(self, store_dir, key):
        path = Path(store_dir) / f"{key}.pt"
        if not path.is_file():
            raise RuntimeError(f"No latent at {path}. Run the sampling stage first.")
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError(f"Unsupported schema: {payload.get('schema_version')}")
        samples = _restore(payload["samples"])
        logging.info("H3_LOAD_LATENT %s", json.dumps(
            {"key": key, "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class H3Fp32TextBranch:
    """Run the text branch in fp32 and pass the MODEL through.

    condition_proj is where fp16 dies: measured input absmax 15424, output inf.
    Only the input needs upcasting -- comfy's cast_bias_weight and ComfyUI-GGUF's
    dequant both target the input dtype, so token_refiner follows into fp32 by
    itself and returns through final_norm (a RMSNorm), which puts the values back
    at O(1) before they are written into the fp16 sequence buffer.

    Video and audio already get this treatment: ComfyUI constructs video_patch_proj
    and audio_patch_proj with dtype=torch.float32. This extends it to text.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {"model": ("MODEL",)}}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "apply"
    CATEGORY = "H3 benchmark/control"

    def apply(self, model):
        target = getattr(getattr(model, "model", None), "diffusion_model", None)
        proj = getattr(target, "condition_proj", None)
        if proj is None:
            logging.warning("H3_FP32_TEXT_BRANCH no condition_proj found; nothing patched")
            return (model,)
        if getattr(proj, "_h3_fp32_text_branch", False):
            logging.info("H3_FP32_TEXT_BRANCH already_applied")
            return (model,)
        original = proj.forward

        def forward(x, *args, **kwargs):
            # bf16 and fp32 already have the range; only fp16 needs this.
            if isinstance(x, torch.Tensor) and x.dtype == torch.float16:
                x = x.to(torch.float32)
            return original(x, *args, **kwargs)

        proj.forward = forward
        proj._h3_fp32_text_branch = True
        logging.info("H3_FP32_TEXT_BRANCH applied to condition_proj "
                     "(token_refiner follows the input dtype)")
        return (model,)


class H3NanTrace:
    """Arm the tracer on a MODEL and pass it straight through."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "model": ("MODEL",),
            "trail_len": ("INT", {"default": 16, "min": 1, "max": 256}),
        }}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "arm"
    CATEGORY = "H3 benchmark/control"

    def arm(self, model, trail_len):
        state = H3_NAN_TRACE
        state.update({"armed": True, "done": False, "order": 0, "trail": [],
                      "trail_len": int(trail_len), "culprit": None, "error": None,
                      "skipped": 0, "errors": []})
        target = getattr(getattr(model, "model", None), "diffusion_model", None)
        if target is None:
            state["error"] = "no diffusion_model on the patcher; nothing hooked"
            logging.warning("H3_NAN_TRACE_ARM %s", state["error"])
            return (model,)
        if getattr(target, "_h3_trace_hooked", False):
            logging.info("H3_NAN_TRACE_ARM already hooked, counters reset")
            return (model,)
        count = 0
        for name, module in target.named_modules():
            module.register_forward_hook(_trace_hook(name))
            count += 1
        target._h3_trace_hooked = True
        state["hooks"] = count
        logging.info("H3_NAN_TRACE_ARM hooked %d modules", count)
        return (model,)


# --- where a step's time goes: attention, MLP, or the allocator ---------------
# The cliff between gear 0.4 and 0.5 is a step in the activations, not a curve
# (per-pixel cost 233 -> 354), and shrinking the DiT by 2.5 GiB moved it by 2%.
# Three candidate mechanisms remain, and they leave different fingerprints in
# the same three measurements:
#   1. per-class CUDA time  -- an attention kernel fallback inflates the
#                              attention classes only, superlinearly
#   2. SDPA call census     -- a chunked/split path changes the call count and
#                              shapes between gears; zero calls means attention
#                              is not going through F.sdpa at all
#   3. allocator counters   -- thrashing shows as alloc retries and device
#                              alloc/free churn, and it inflates EVERY class
# Cost: two CUDA events per hooked call and ONE synchronize per model forward.
# A step is minutes; the probe is milliseconds. Probed timings stay measurements.
H3_PERF = {"armed": False, "forwards": 0, "hooks": 0, "events": [], "sdpa": {},
           "sdpa_events": [], "backend_cache": {}, "steps": [], "error": None,
           "pre_counters": None, "pre_t": None, "root_events": None,
           "rebound": 0, "allocator_backend": None}

_PERF_CLASS_RE = re.compile(r"attention|attn|mlp|feedforward|ffn", re.I)
_SDP_BACKEND_NAMES = {-1: "ERROR", 0: "MATH", 1: "FLASH_ATTENTION",
                      2: "EFFICIENT_ATTENTION", 3: "CUDNN_ATTENTION",
                      4: "OVERRIDEABLE"}


def _perf_counters():
    """Allocator event counters, for deltas across one forward.

    No reset_peak_memory_stats: ComfyUI reads the same accounting for its own
    load/unload decisions, so peaks are reported monotonically and the per-step
    signal lives in these monotone counters instead.
    """
    try:
        stats = torch.cuda.memory_stats()
    except Exception:
        return {}
    return {key: int(stats.get(key, 0)) for key in (
        "num_alloc_retries", "num_ooms", "num_device_alloc", "num_device_free")}


def _perf_event():
    event = torch.cuda.Event(enable_timing=True)
    event.record()
    return event


def _perf_pre_module(module, args):
    if not H3_PERF["armed"]:
        return
    try:
        stack = getattr(module, "_h3_perf_stack", None)
        if stack is None:
            stack = []
            module._h3_perf_stack = stack
        stack.append(_perf_event())
    except Exception:
        pass


def _perf_post_module(module, args, output):
    state = H3_PERF
    if not state["armed"]:
        return
    try:
        stack = getattr(module, "_h3_perf_stack", None)
        if not stack:
            return
        begin = stack.pop()
        state["events"].append((type(module).__name__, begin, _perf_event()))
    except Exception:
        pass


def _perf_sdpa_wrapper(original):
    def wrapped(query, key, value, *args, **kwargs):
        state = H3_PERF
        if (not state["armed"] or not isinstance(query, torch.Tensor)
                or not query.is_cuda):
            return original(query, key, value, *args, **kwargs)
        begin = signature = None
        try:
            mask = kwargs.get("attn_mask", args[0] if args else None)
            dropout = kwargs.get("dropout_p", args[1] if len(args) > 1 else 0.0)
            causal = bool(kwargs.get("is_causal",
                                     args[2] if len(args) > 2 else False))
            signature = (f"q{tuple(query.shape)} kv{tuple(key.shape)} "
                         f"{str(query.dtype).replace('torch.', '')} "
                         f"causal={causal} mask={mask is not None}")
            if signature not in state["backend_cache"]:
                # What the dispatcher WOULD pick for this exact call. Cached per
                # signature; the answer cannot change for identical inputs.
                try:
                    choice = torch._fused_sdp_choice(query, key, value, mask,
                                                     dropout, causal)
                    state["backend_cache"][signature] = _SDP_BACKEND_NAMES.get(
                        int(choice), f"UNKNOWN_{choice}")
                except Exception as exc:
                    state["backend_cache"][signature] = \
                        f"choice_unavailable:{type(exc).__name__}"
            state["sdpa"].setdefault(signature,
                                     {"calls": 0, "ms": 0.0})["calls"] += 1
            begin = _perf_event()
        except Exception:
            begin = None
        out = original(query, key, value, *args, **kwargs)
        if begin is not None:
            try:
                state["sdpa_events"].append((signature, begin, _perf_event()))
            except Exception:
                pass
        return out
    wrapped._h3_perf_original = original
    return wrapped


def _perf_install_sdpa():
    """Wrap F.scaled_dot_product_attention and rebind every module-level alias.

    Modules that did `from torch.nn.functional import scaled_dot_product_attention`
    hold their own reference, so patching the F attribute alone misses them. The
    scan replaces any module attribute that IS the original function object,
    which is semantically transparent. If the census still records zero calls,
    that is itself the answer: attention runs through a fused non-SDPA path.
    """
    original = torch.nn.functional.scaled_dot_product_attention
    if getattr(original, "_h3_perf_original", None) is not None:
        return 0                                   # already wrapped
    wrapped = _perf_sdpa_wrapper(original)
    rebound = 0
    for _name, module in list(sys.modules.items()):
        if module is None:
            continue
        try:
            for attr, value in list(vars(module).items()):
                if value is original:
                    setattr(module, attr, wrapped)
                    rebound += 1
        except Exception:
            continue
    return rebound


def _perf_pre_root(module, args):
    state = H3_PERF
    if not state["armed"]:
        return
    try:
        state["pre_counters"] = _perf_counters()
        state["pre_t"] = time.perf_counter()
        state["events"] = []
        state["sdpa_events"] = []
        state["root_events"] = (_perf_event(),
                                torch.cuda.Event(enable_timing=True))
    except Exception as exc:
        state["root_events"] = None
        state["error"] = f"pre_root {type(exc).__name__}: {str(exc)[:200]}"


def _perf_post_root(module, args, output):
    state = H3_PERF
    if not state["armed"] or state["root_events"] is None:
        return
    try:
        begin, end = state["root_events"]
        state["root_events"] = None
        end.record()
        torch.cuda.synchronize()               # the one sync this probe costs
        total_ms = begin.elapsed_time(end)
        per_class = {}
        for cls, b, e in state["events"]:
            entry = per_class.setdefault(cls, {"ms": 0.0, "calls": 0})
            entry["ms"] += b.elapsed_time(e)
            entry["calls"] += 1
        hooked_ms = sum(v["ms"] for v in per_class.values())
        for entry in per_class.values():
            entry["ms"] = round(entry["ms"], 1)
        sdpa_ms = 0.0
        for signature, b, e in state["sdpa_events"]:
            elapsed = b.elapsed_time(e)
            sdpa_ms += elapsed
            record = state["sdpa"].setdefault(signature,
                                              {"calls": 0, "ms": 0.0})
            record["ms"] += elapsed
        post = _perf_counters()
        pre = state["pre_counters"] or {}
        try:
            stats = torch.cuda.memory_stats()
            memory = {"allocated_mib": round(
                          stats.get("allocated_bytes.all.current", 0) / 2**20),
                      "reserved_mib": round(
                          stats.get("reserved_bytes.all.current", 0) / 2**20),
                      "allocated_peak_mib": round(
                          stats.get("allocated_bytes.all.peak", 0) / 2**20)}
        except Exception:
            memory = {}
        state["forwards"] += 1
        step = {"forward": state["forwards"],
                "wall_s": (round(time.perf_counter() - state["pre_t"], 2)
                           if state["pre_t"] else None),
                "cuda_total_ms": round(total_ms, 1),
                "per_class_ms": per_class,
                "unhooked_ms": round(total_ms - hooked_ms, 1),
                "sdpa_calls": len(state["sdpa_events"]),
                "sdpa_ms": round(sdpa_ms, 1),
                "alloc_delta": {key: post.get(key, 0) - pre.get(key, 0)
                                for key in post}}
        step.update(memory)
        state["steps"].append(step)
        if len(state["steps"]) > 64:
            state["steps"].pop(0)
        state["events"] = []
        state["sdpa_events"] = []
        logging.info("H3_PERF_STEP %s", json.dumps(step, separators=(",", ":")))
    except Exception as exc:
        state["error"] = f"post_root {type(exc).__name__}: {str(exc)[:200]}"


def _perf_report():
    state = H3_PERF
    if not state["armed"]:
        return None
    census = [{"sig": sig, "backend": state["backend_cache"].get(sig),
               "calls": entry["calls"], "ms": round(entry["ms"], 1)}
              for sig, entry in sorted(state["sdpa"].items())]
    return {"armed": True, "hooks": state["hooks"],
            "forwards": state["forwards"],
            "sdpa_refs_rebound": state["rebound"],
            "allocator_backend": state["allocator_backend"],
            "sdpa_census": census, "steps": state["steps"],
            "error": state["error"],
            "note": ("per_class_ms is CUDA time by module class; a kernel "
                     "fallback inflates attention only, allocator thrashing "
                     "raises alloc_delta counters, a split path changes sdpa "
                     "call count/shapes, sdpa_calls=0 means a fused non-SDPA "
                     "path")}


class H3PerfProbe:
    """Arm the per-step performance probe on a MODEL and pass it through.

    Same shape as H3NanTrace: a pass-through on MODEL that instruments the
    diffusion model underneath. The three measurements together separate the
    cliff candidates; see the comment block above H3_PERF.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {"model": ("MODEL",)}}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "arm"
    CATEGORY = "H3 benchmark/control"

    def arm(self, model):
        state = H3_PERF
        state.update({"armed": True, "forwards": 0, "events": [], "sdpa": {},
                      "sdpa_events": [], "backend_cache": {}, "steps": [],
                      "error": None, "pre_counters": None, "pre_t": None,
                      "root_events": None})
        try:
            state["allocator_backend"] = torch.cuda.get_allocator_backend()
        except Exception:
            state["allocator_backend"] = "unknown"
        state["rebound"] = _perf_install_sdpa()
        target = getattr(getattr(model, "model", None), "diffusion_model", None)
        if target is None:
            state["error"] = "no diffusion_model on the patcher; nothing hooked"
            logging.warning("H3_PERF_ARM %s", state["error"])
            return (model,)
        if getattr(target, "_h3_perf_hooked", False):
            logging.info("H3_PERF_ARM already hooked, counters reset")
            return (model,)
        target.register_forward_pre_hook(_perf_pre_root)
        target.register_forward_hook(_perf_post_root)
        selected = []
        classes = set()
        for name, module in target.named_modules():
            if not name or not _PERF_CLASS_RE.search(type(module).__name__):
                continue
            # named_modules yields parents before children; skipping descendants
            # of an already-timed module keeps the per-class sums disjoint.
            if any(name.startswith(prefix + ".") for prefix in selected):
                continue
            selected.append(name)
            classes.add(type(module).__name__)
            module.register_forward_pre_hook(_perf_pre_module)
            module.register_forward_hook(_perf_post_module)
        target._h3_perf_hooked = True
        state["hooks"] = len(selected)
        logging.info(
            "H3_PERF_ARM hooked %d modules (classes %s), rebound %d sdpa refs, "
            "allocator %s", len(selected), ",".join(sorted(classes)),
            state["rebound"], state["allocator_backend"])
        return (model,)



H3_RMSNORM_FP32 = __RMSNORM_FP32__


def _install_fp32_rmsnorm():
    """Run RMSNorm in fp32 without giving up fp16 tensor cores elsewhere.

    ComfyUI-GGUF never overrides RMSNorm, so replacing the method on
    disable_weight_init reaches every RMSNorm the GGUF path uses. The fused
    q_norm/k_norm inside Attention calls comfy_kitchen directly and is not covered.
    """
    import comfy.ops as ops

    target = ops.disable_weight_init.RMSNorm
    if getattr(target, "_h3_fp32_rmsnorm", False):
        return "already_applied"

    def forward_comfy_cast_weights(self, input):
        if self.weight is not None:
            weight, bias, offload_stream = ops.cast_bias_weight(
                self, input, offloadable=True)
        else:
            weight = bias = offload_stream = None
        if input.dtype == torch.float16:
            # bf16 and fp32 already have the exponent range; only fp16 needs this.
            x = torch.nn.functional.rms_norm(
                input.float(), self.normalized_shape,
                weight.float() if weight is not None else None,
                self.eps).to(input.dtype)
        else:
            x = torch.nn.functional.rms_norm(
                input, self.normalized_shape, weight, self.eps)
        uncast = getattr(ops, "uncast_bias_weight", None)
        if uncast is not None:
            uncast(self, weight, bias, offload_stream)
        return x

    target.forward_comfy_cast_weights = forward_comfy_cast_weights
    target._h3_fp32_rmsnorm = True
    return "applied"


if H3_RMSNORM_FP32:
    try:
        logging.info("H3_FP32_RMSNORM %s (fused q/k rope path is NOT covered)",
                     _install_fp32_rmsnorm())
    except Exception as exc:                                  # never fatal
        logging.warning("H3_FP32_RMSNORM failed: %s: %s", type(exc).__name__, exc)


NODE_CLASS_MAPPINGS = {
    "H3SaveConditioning": H3SaveConditioning,
    "H3LoadConditioning": H3LoadConditioning,
    "H3LatentProbe": H3LatentProbe,
    "H3NanTrace": H3NanTrace,
    "H3PerfProbe": H3PerfProbe,
    "H3Fp32TextBranch": H3Fp32TextBranch,
    "H3SaveLatent": H3SaveLatent,
    "H3LoadLatent": H3LoadLatent,
}
NODE_DISPLAY_NAME_MAPPINGS = {
    "H3SaveConditioning": "H3 Save Precomputed Conditioning",
    "H3LoadConditioning": "H3 Load Precomputed Conditioning",
    "H3LatentProbe": "H3 Sampled Latent Probe",
    "H3NanTrace": "H3 First Non-Finite Module Trace",
    "H3PerfProbe": "H3 Per-Step Performance Probe",
    "H3Fp32TextBranch": "H3 fp32 Text Branch",
    "H3SaveLatent": "H3 Save Sampled Latent",
    "H3LoadLatent": "H3 Load Sampled Latent",
}
'''.replace("__RMSNORM_FP32__", repr(RMSNORM_FP32)), encoding="utf-8")
print("自建节点已写入", (CTRL / "__init__.py").stat().st_size, "bytes")

# ---- ComfyUI 进程控制（极简）----
_COMFY_PROCESS = None
def stop_comfy():
    global _COMFY_PROCESS
    if _COMFY_PROCESS and _COMFY_PROCESS.poll() is None:
        _COMFY_PROCESS.terminate()
        try: _COMFY_PROCESS.wait(timeout=15)
        except Exception: _COMFY_PROCESS.kill()
    _COMFY_PROCESS = None

def start_comfy(dtype_flag="--bf16-unet", timeout=600):
    global _COMFY_PROCESS
    stop_comfy()
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--disable-auto-launch", "--preview-method", "none",
           "--output-directory", str(OUTPUT_DIR)]
    if dtype_flag:
        cmd.append(dtype_flag)
    h = open(COMFY_LOG, "a", buffering=1)
    _COMFY_PROCESS = subprocess.Popen(cmd, cwd=str(COMFY_DIR), stdout=h, stderr=subprocess.STDOUT)
    for _ in range(timeout // 2):
        if _COMFY_PROCESS.poll() is not None:
            raise RuntimeError("ComfyUI 启动退出:\n" + COMFY_LOG.read_text()[-3000:])
        s = socket.socket(); ok = s.connect_ex(("127.0.0.1", 8188)) == 0; s.close()
        if ok: return
        time.sleep(2)
    raise RuntimeError("ComfyUI 启动超时")

# ---- cloudflared（Cell 4 隧道需要；静态二进制直装 /usr/local/bin，不走 dpkg）----
if shutil.which("cloudflared") is None:
    _sh(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    os.chmod("/usr/local/bin/cloudflared", 0o755)
if shutil.which("cloudflared") is None:   # 装完必须真的在，否则大声报错（不静默糊弄过去）
    raise RuntimeError(
        "cloudflared 安装失败！请手动执行：\n"
        "!wget -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\n"
        "!chmod +x /usr/local/bin/cloudflared\n"
        "然后重跑本 Cell。通常是 GitHub 访问抖动，多试一次即可。")
print("cloudflared:", shutil.which("cloudflared"))

print("安装完成")




In [ ]:
# @title 2. 下载权重（首次约 30G，10-20 分钟）
import os
from huggingface_hub import hf_hub_download, login

tok = os.environ.get("HF_TOKEN") or ""
if tok:
    login(token=tok, add_to_git_credential=False)

def _link(local, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        if dest.is_symlink() and dest.resolve() == local.resolve():
            return
        dest.unlink()
    dest.symlink_to(local)

FILES = [
    ("VideoRebirthAI/hyperflow-h3-flash", "fl2va_gguf/minimax_h3_fl2va_hyperflow_flash_14b-Q4_K_M.gguf", "diffusion_models"),
    ("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors", "vae"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
]

for repo, fname, sub in FILES:
    # hf_hub_download 会保留 filename 的子目录层级（如 vae/xxx），所以按两个候选位置找，
    # 再用递归兜底；找到真实文件后才建链接，避免死链
    name = Path(fname).name
    def _find():
        for c in (MODEL_STORE / sub / name, MODEL_STORE / sub / fname):
            if c.is_file():
                return c
        hits = [q for q in MODEL_STORE.rglob(name) if q.is_file() and ".huggingface" not in str(q)]
        return hits[0] if hits else None
    local = _find()
    if local is None:
        print(f"下载 {repo}/{fname}")
        hf_hub_download(repo_id=repo, filename=fname, local_dir=str(MODEL_STORE / sub))
        local = _find()
        assert local is not None, f"下载后仍未找到 {fname}，请检查 {MODEL_STORE}"
    _link(local, COMFY_DIR / "models" / sub / name)
    print("  OK ->", local)

tot = sum(p.stat().st_size for p in MODEL_STORE.rglob("*") if p.is_file())
print(f"模型合计 {tot / 1e9:.1f} GB")
print("下载完成")



In [ ]:
# @title 3. 启动简单界面 + API（输入提示词出 HyperFlow 视频）
import requests, uuid, threading, time, socket, secrets, hashlib
import base64, io
from typing import Optional
from PIL import Image
from pathlib import Path
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import FileResponse, HTMLResponse
from pydantic import BaseModel
import uvicorn

# ---- 生成参数（改这里即可）----
GEAR_W, GEAR_H = 512, 384     # 32 的倍数；T4 上别超过 768 宽，容易 OOM
CLIP_SECONDS = 2.0            # 2.0→56 帧(约2.3秒)；5.2→124 帧，耗时约×2.2；上限 15 秒
STEPS = 6                      # HyperFlow 只支持 4(快)/6(默认)/8(画质)，无 CFG
FPS = 24

TEXT_ENC = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
VIDEO_VAE = "minimax_h3_video_vae_fp16.safetensors"
AUDIO_VAE = "minimax_h3_audio_vae_fp32.safetensors"
HF_GGUF = "minimax_h3_fl2va_hyperflow_flash_14b-Q4_K_M.gguf"

def _frames(seconds):
    n = max(5, int(seconds * FPS))   # int 取整：5.2 秒→124 帧（round 会得到 141 帧白跑）
    return n + (5 - (n % 17)) % 17   # 帧数须满足 17n+5

def _key(prompt, img_hash="", w=None, h=None):
    w = w or GEAR_W; h = h or GEAR_H
    return hashlib.md5(f"{w}x{h}x{_frames(CLIP_SECONDS)}x{STEPS}|{img_hash}|{prompt}".encode()).hexdigest()[:12]

# ---- ComfyUI 提交 / 轮询 ----
def _submit(graph):
    r = requests.post(f"{COMFY_API}/prompt", timeout=120,
                      json={"prompt": graph, "client_id": uuid.uuid4().hex})
    r.raise_for_status()
    if r.json().get("node_errors"):
        raise RuntimeError(f"节点错误: {r.json()['node_errors']}")
    return r.json()["prompt_id"]

def _wait_done(pid, timeout, poll=5):
    t0 = time.time()
    while time.time() - t0 < timeout:
        h = requests.get(f"{COMFY_API}/history/{pid}", timeout=60).json()
        if pid in h:
            st = h[pid].get("status", {})
            if st.get("completed") or st.get("status_str") == "error":
                if st.get("status_str") != "success":
                    raise RuntimeError(f"执行失败: {st}")
                return h[pid]
        time.sleep(poll)
    raise TimeoutError("执行超时")

# ---- 三阶段（各独立进程，峰值=max(stage)）----
def encode_stage(key, prompt, img_file=None, w=None, h=None):
    start_comfy("--bf16-unet")
    W, H = (w or GEAR_W), (h or GEAR_H)
    g = {
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": TEXT_ENC, "type": "minimax", "device": "default"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": VIDEO_VAE}},
        "5": {"class_type": "MiniMaxH3ImageToVideo", "inputs": {
            "clip": ["2", 0], "vae": ["3", 0], "prompt": prompt,
            "width": W, "height": H, "length": _frames(CLIP_SECONDS)}},
        "40": {"class_type": "H3SaveConditioning", "inputs": {
            "conditioning": ["5", 0], "samples": ["5", 1],
            "store_dir": str(COND_STORE), "key": key}},
    }
    if img_file is not None:   # i2v：官方链路 LoadImage → first_frame
        g["20"] = {"class_type": "LoadImage", "inputs": {"image": img_file}}
        g["5"]["inputs"]["first_frame"] = ["20", 0]
    _wait_done(_submit(g), 3600)
    stop_comfy()

def sample_stage(key):
    start_comfy()  # --bf16-unet（H3 家族 fp16 会 NaN 溢出，须 bf16）
    g = {
        "1": {"class_type": "HyperFlowLoader", "inputs": {"unet_name": HF_GGUF, "weights_in": "disk"}},
        "61": {"class_type": "HyperFlowSigmas", "inputs": {"model": ["1", 0], "steps": STEPS}},
        "7": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}},
        "41": {"class_type": "H3LoadConditioning", "inputs": {"store_dir": str(COND_STORE), "key": key}},
        "6": {"class_type": "RandomNoise", "inputs": {"noise_seed": 42}},
        "9": {"class_type": "BasicGuider", "inputs": {"model": ["1", 0], "conditioning": ["41", 0]}},
        "10": {"class_type": "SamplerCustomAdvanced", "inputs": {
            "noise": ["6", 0], "guider": ["9", 0], "sampler": ["7", 0],
            "sigmas": ["61", 0], "latent_image": ["41", 1]}},
        "18": {"class_type": "H3SaveLatent", "inputs": {"samples": ["10", 0], "store_dir": str(LATENT_STORE), "key": key}},
        "15": {"class_type": "H3LatentProbe", "inputs": {"samples": ["18", 0], "filename_prefix": f"h3_sample/{key}"}},
    }
    _wait_done(_submit(g), 7200)
    stop_comfy()

def decode_stage(key):
    start_comfy(dtype_flag=None)  # 无 DiT，不需要 bf16
    g = {
        "1": {"class_type": "VAELoader", "inputs": {"vae_name": VIDEO_VAE}},
        "2": {"class_type": "VAELoader", "inputs": {"vae_name": AUDIO_VAE}},
        "3": {"class_type": "H3LoadLatent", "inputs": {"store_dir": str(LATENT_STORE), "key": key}},
        "4": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["1", 0]}},
        "5": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["3", 0], "vae": ["2", 0]}},
        "6": {"class_type": "CreateVideo", "inputs": {"images": ["4", 0], "audio": ["5", 0], "fps": float(FPS)}},
        "7": {"class_type": "SaveVideo", "inputs": {"video": ["6", 0], "filename_prefix": f"h3_video/{key}", "format": "auto", "codec": "auto"}},
    }
    rec = _wait_done(_submit(g), 1800)
    stop_comfy()
    for o in rec.get("outputs", {}).values():
        for item in o.get("gifs", []) + o.get("videos", []) + o.get("images", []):
            if item.get("filename"):
                return str(OUTPUT_DIR / item.get("subfolder", "") / item["filename"])
    out = sorted((OUTPUT_DIR / "h3_video" / key).glob("*"))
    if out: return str(out[-1])
    raise RuntimeError("未找到产物 MP4")

def generate(prompt, img_file=None, w=None, h=None, img_hash=""):
    key = _key(prompt, img_hash, w, h)
    encode_stage(key, prompt, img_file, w, h)
    sample_stage(key)
    return decode_stage(key)

# ---- FastAPI + 简单界面 ----
API_KEY = "h3-" + secrets.token_hex(4)
JOBS = {}

class GenReq(BaseModel):
    prompt: str
    image_b64: Optional[str] = None   # 可选参考图（PNG/JPEG base64）；带上即走 i2v

_GPU_LOCK = threading.Lock()   # 单卡串行：三阶段共用同一个 ComfyUI 进程，不能并发

def _run_job(job_id, prompt, img_b64=None):
    try:
        img_file, w, h, img_hash = None, None, None, ""
        if img_b64:
            JOBS[job_id]["status"] = "处理参考图…"
            im = Image.open(io.BytesIO(base64.b64decode(img_b64))).convert("RGB")
            iw, ih_ = im.size
            ratio = iw / ih_
            short = min(GEAR_W, GEAR_H)          # 沿用默认算力预算的短边
            if ratio >= 1.0:
                h = short; w = min(768, max(short, round(short * ratio / 32) * 32))
            else:
                w = short; h = min(768, max(short, round(short / ratio / 32) * 32))
            fn = f"job_{job_id}.png"
            _p = COMFY_DIR / "input" / fn
            _p.parent.mkdir(parents=True, exist_ok=True)
            im.save(_p, "PNG")
            img_file, img_hash = fn, hashlib.md5(img_b64.encode()).hexdigest()[:10]
        with _GPU_LOCK:
            JOBS[job_id]["status"] = "排队/encode（首次还要下载15.7G编码器）"
            encode_stage(_key(prompt, img_hash, w, h), prompt, img_file, w, h)
            JOBS[job_id]["status"] = "sample（T4 上约 15-40 分钟，视分辨率）"
            sample_stage(_key(prompt, img_hash, w, h))
            JOBS[job_id]["status"] = "decode"
            mp4 = decode_stage(_key(prompt, img_hash, w, h))
        JOBS[job_id].update(status="done", file=mp4)
    except Exception as exc:
        JOBS[job_id].update(status="failed", error=f"{type(exc).__name__}: {exc}")

app = FastAPI(title="HyperFlow H3 Flash 简单界面", docs_url="/docs")

def _check(request):
    if request.headers.get("x-api-key") != API_KEY:
        raise HTTPException(401, "无效 API Key")

@app.get("/api/health")
def health(): return {"status": "ok"}

@app.post("/api/jobs")
async def create(request: Request, body: GenReq):
    _check(request)
    if not body.prompt.strip():
        raise HTTPException(400, "prompt 不能为空")
    if body.image_b64 and len(body.image_b64) > 15 * 1024 * 1024:
        raise HTTPException(400, "图片过大（约>11MB），请压缩后再传")
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"status": "queued", "prompt": body.prompt, "i2v": bool(body.image_b64)}
    threading.Thread(target=_run_job, args=(job_id, body.prompt, body.image_b64), daemon=True).start()
    return {"job_id": job_id, "status_url": f"/api/jobs/{job_id}",
            "file_url": f"/api/jobs/{job_id}/file"}

@app.get("/api/jobs/{job_id}")
def job_status(job_id: str):
    if job_id not in JOBS: raise HTTPException(404, "未知 job")
    return JOBS[job_id]

@app.get("/api/jobs/{job_id}/file")
def job_file(job_id: str):
    j = JOBS.get(job_id)
    if not j or j["status"] != "done": raise HTTPException(404, "尚未完成")
    return FileResponse(j["file"], media_type="video/mp4")

PAGE = r"""<!doctype html><html lang="zh"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>HyperFlow H3 Flash 视频生成</title><style>
body{font-family:system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;background:#f3f4f6;margin:0;padding:24px;color:#111827}
.wrap{max-width:760px;margin:0 auto}
h1{font-size:22px;margin:0 0 16px}
.card{background:#fff;border:1px solid #e5e7eb;border-radius:12px;padding:20px;margin-bottom:14px;box-shadow:0 1px 3px rgba(0,0,0,.06)}
label{display:block;font-size:13px;color:#374151;margin:10px 0 4px}
textarea,input{width:100%;padding:9px;border:1px solid #d1d5db;border-radius:8px;box-sizing:border-box;font-size:14px}
textarea{min-height:90px;resize:vertical}
button{margin-top:14px;width:100%;padding:12px;background:#2563eb;color:#fff;border:0;border-radius:8px;font-size:15px;cursor:pointer}
button:disabled{background:#9ca3af;cursor:not-allowed}
#log{font-family:ui-monospace,Menlo,monospace;font-size:12px;white-space:pre-wrap;background:#111827;color:#34d399;padding:12px;border-radius:8px;min-height:60px;max-height:220px;overflow:auto}
video{max-width:100%;border-radius:8px;margin-top:8px}
.hint{font-size:12px;color:#6b7280;margin-top:8px;line-height:1.6}
</style></head><body><div class="wrap">
<h1>HyperFlow H3 Flash 视频生成</h1>
<div class="card">
<label>API Key（自动填好）</label><input id="key" type="text" value="__KEY__" readonly>
<label>参考图片（可选）：带上后按图片比例出视频（i2v），首帧即此图</label>
<input type="file" id="img" accept="image/png,image/jpeg,image/webp" onchange="pickImg(this)">
<img id="imgprev" alt="" style="display:none;max-height:140px;border-radius:8px;border:1px solid #d1d5db;margin-top:8px">
<label>提示词</label>
<textarea id="prompt" placeholder="描述画面和声音，中文即可，例如：夜晚城市街道车流，霓虹灯光，慢镜头，雨声与远处车流声"></textarea>
<button id="go" onclick="submitJob()">开始生成（默认约25-45分钟）</button>
<div class="hint">免费 T4 + HyperFlow H3 Flash（13.7B 压缩版，4-8 步采样，同步生成 32kHz 立体声）。默认一段 2.3 秒视频约 25-45 分钟，出片后自动播放（含音轨，记得开声音）。上传参考图即走 i2v：输出按图片比例适配（短边固定，宽/高≤768）。会话断了就回 Colab 重跑 Cell 3、4 换新地址。</div>
</div>
<div class="card"><div id="log">等待提交…</div></div>
<div class="card"><div id="result"></div></div>
</div>
<script>
function log(t){const d=document.getElementById('log');d.textContent+=('['+new Date().toLocaleTimeString()+'] '+t+'\n');d.scrollTop=d.scrollHeight}
function pickImg(inp){
  const prev=document.getElementById('imgprev');
  if(inp.files&&inp.files[0]){
    const rd=new FileReader();
    rd.onload=e=>{prev.src=e.target.result;prev.style.display='block'};
    rd.readAsDataURL(inp.files[0]);
    log('已选择参考图：'+inp.files[0].name+'（将按图片比例生成）');
  }else{prev.style.display='none';}
}
async function submitJob(){
  const key=document.getElementById('key').value;
  const prompt=document.getElementById('prompt').value.trim();
  if(!prompt){log('请先填提示词');return}
  const f=document.getElementById('img').files[0];
  let imgb64=null;
  if(f){
    if(f.size>10*1024*1024){log('图片过大（>10MB），请压缩后重选');return}
    imgb64=await new Promise(r=>{const rd=new FileReader();rd.onload=()=>r(rd.result.split(',')[1]);rd.readAsDataURL(f);});
  }
  const btn=document.getElementById('go');btn.disabled=true;btn.textContent='生成中（约25-45分钟）…';
  try{
    const r=await fetch('/api/jobs',{method:'POST',headers:{'Content-Type':'application/json','x-api-key':key},body:JSON.stringify({prompt:prompt,image_b64:imgb64})});
    const j=await r.json();
    if(!r.ok)throw new Error('HTTP '+r.status+' '+(j.detail||JSON.stringify(j)));
    log('已提交 job='+j.job_id+'，开始轮询（请耐心等待）…');
    await poll(j.job_id,key);
  }catch(e){log('错误: '+e.message)}
  btn.disabled=false;btn.textContent='开始生成';
}
async function poll(id,key){
  for(let i=0;i<900;i++){
    await new Promise(r=>setTimeout(r,6000));
    try{
      const r=await fetch('/api/jobs/'+id,{headers:{'x-api-key':key}});
      const s=await r.json();
      log(s.status+(s.error?(' | '+s.error):''));
      if(s.status==='done'){document.getElementById('result').innerHTML='<video controls autoplay src="/api/jobs/'+id+'/file"></video>';return;}
      if(s.status==='failed')return;
    }catch(e){log('轮询异常: '+e.message)}
  }
  log('超时退出轮询（视频可能仍需一会）');
}
</script></body></html>"""
PAGE = PAGE.replace("__KEY__", API_KEY)

@app.get("/", response_class=HTMLResponse)
def home(): return PAGE

# ---- 启动后台服务 ----
_s = socket.socket(); _busy = _s.connect_ex(("127.0.0.1", 8000)) == 0; _s.close()
if _busy:
    print("⚠️ 警告：8000 端口已有服务在运行。请不要重复运行本 Cell；")
    print("   如需重启，请先『重启运行时』，再从 Cell 1 重新跑（否则新 Key 与旧服务不一致，网页会像'没反应'）。")
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
                 daemon=True).start()
for _ in range(60):
    s = socket.socket(); ok = s.connect_ex(("127.0.0.1", 8000)) == 0; s.close()
    if ok: break
    time.sleep(1)
print("本机服务: 127.0.0.1:8000")
print("API Key:", API_KEY)



In [ ]:
# @title 4. 拉起公网隧道域名（跑完看下面；浏览器用 http:// 打开）
import subprocess, time, re, socket, os, shutil

TLOG = "/content/tunnel.log"
LOCAL = "http://127.0.0.1:8000"

def _open(port):
    s = socket.socket(); ok = s.connect_ex(("127.0.0.1", port)) == 0; s.close(); return ok

# cloudflared 自举：不依赖 Cell 1 是否装过——缺了就地补装（静态二进制，不走 dpkg）
def _ensure_cloudflared():
    if shutil.which("cloudflared"):
        return True
    try:
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                       check=True)
        os.chmod("/usr/local/bin/cloudflared", 0o755)
    except Exception as e:
        print("下载 cloudflared 异常:", e)
    return shutil.which("cloudflared") is not None

def _find_pub():
    try:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", open(TLOG).read())
        return m.group(0) if m else None
    except FileNotFoundError:
        return None

def _start_tunnel():
    # 只清理本笔记本自己启动过的旧 cloudflared，避免重复运行 Cell 4 时留下多个隧道进程
    subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
    time.sleep(1)
    tlog = open(TLOG, "w")
    subprocess.Popen(["cloudflared", "tunnel", "--url", LOCAL],
                     stdout=tlog, stderr=subprocess.STDOUT)

PUB = None
if _ensure_cloudflared():
    for attempt in (1, 2):
        _start_tunnel()
        deadline = time.time() + 90
        while time.time() < deadline:
            time.sleep(2)
            PUB = _find_pub()
            if PUB:
                break
        if PUB:
            break
        print(f"第 {attempt} 次尝试 90 秒内未拿到域名，重试一次…")
else:
    print("❌ cloudflared 未能就绪。手动执行下面两行，然后重跑本 Cell：")
    print("   !wget -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    print("   !chmod +x /usr/local/bin/cloudflared")

print()
if PUB:
    print("Web 界面(浏览器用 http 打开):", PUB.replace("https://", "http://") + "/")
    print("API 根地址:", PUB)
else:
    print("未拿到隧道域名。排查：在下面输入 !cat /content/tunnel.log 看报错；")
    print("并确认『本机 8000』为在线（未在线就先跑 Cell 3 再回来重跑本 Cell）。")
print("API Key    :", API_KEY)
print("本机 8000  :", "在线" if _open(8000) else "未监听（先回 Cell 3 启动服务再跑本 Cell）")
print("提示：域名每次重跑 Cell 4 都会换；浏览器请用 http:// 而不是 https://")


In [ ]:
# @title 5. 服务自检 + 防闲置（不提交任何生成任务）
import requests, time
API_BASE = "http://127.0.0.1:8000"   # Colab 内部测试直连本机；外部程序才用 Cell 4 打印的 PUB 公网地址
# （Colab 虚拟机经常解析不了刚分配的 trycloudflare 域名，从内部绕公网会 DNS 报错）
HEAD = {"x-api-key": API_KEY}
print("健康:", requests.get(f"{API_BASE}/api/health", timeout=30).json())

# 本 Cell 不提交任何生成任务（避免空耗 GPU）。出片请在网页里输入提示词点「开始生成」。
print("本 Cell 只做服务自检，不提交任何生成任务。请打开 Cell 4 打印的网页地址(http://)使用。")

from IPython.display import Javascript
Javascript("""
function ConnectButton(){var b=document.querySelector('colab-connect-button');if(b&&b.shadowRoot)b.shadowRoot.querySelector('#connect').click();}
setInterval(ConnectButton,60000);
""")
print("保活 JS 已注入")


